# C1 Placement Ablation — Multi-Model Analysis (benchmark v2)

Analyzes the three completed C1 rerun arms (72 items × 6 variants × 3 agent models, judged by the
**fixed** `gemini-flash-latest` judge):

| arm | dir | agent model |
|---|---|---|
| Gemma 4 31B-IT | `outputs/placement_ablation_v2_gemma-4-31b-it` | open 31B |
| Gemini 3.6 Flash | `outputs/placement_ablation_v2_gemini-3.6-flash` | frontier flash |
| Gemini 3.1 Pro (preview) | `outputs/placement_ablation_v2_gemini-3.1-pro-preview` | frontier large |

Variants: V0 single raw-query search · V1 generic 4-query fanout · V2 persona at synthesis only ·
V3 persona in fanout only · V4 persona in both · V5 mixed fanout (+ persona at synthesis).

All figures are written to `reports/paper/figures/` (PDF for LaTeX + PNG preview) and tables to
`reports/paper/tables/`. Re-run end to end with **Run All** — everything is deterministic (seeded
bootstraps, no API calls).

In [1]:

import json, math
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
FIG_DIR = ROOT / "reports" / "paper" / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR = ROOT / "reports" / "paper" / "tables";  TAB_DIR.mkdir(parents=True, exist_ok=True)

# --- palette (validated defaults; identical to scripts/make_paper_figures.py) ---
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e7e6e2"
MODELS = {  # display order = capability order; color follows the entity in every figure
    "gemma-4-31b-it":          dict(label="Gemma 4 31B-IT",         short="Gemma 4 31B",     color="#2a78d6"),
    "gemini-3.6-flash":        dict(label="Gemini 3.6 Flash",       short="3.6 Flash",       color="#008300"),
    "gemini-3.1-pro-preview":  dict(label="Gemini 3.1 Pro (prev.)", short="3.1 Pro (prev.)", color="#e87ba4"),
}
CH_EVIDENCE, CH_SYNTH = "#1baf7a", "#eb6834"   # the two personalization channels
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#f7fafd", "#7db2e8", "#1c5aa8"])

VKEY = {"V0_generic_single": "V0", "V1_generic_fanout": "V1",
        "V2_synthesis_only_personalization": "V2", "V3_fanout_only_personalization": "V3",
        "V4_personalized_fanout": "V4", "V5_mixed_fanout": "V5"}
V = ["V0", "V1", "V2", "V3", "V4", "V5"]
VLAB = {"V0": "V0\nsingle", "V1": "V1\ngeneric", "V2": "V2\npersona@\nsynth",
        "V3": "V3\npersona@\nfanout", "V4": "V4\nboth", "V5": "V5\nmixed"}

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "axes.edgecolor": INK2, "axes.linewidth": 0.8, "axes.labelcolor": INK,
    "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.7,
    "axes.axisbelow": True, "legend.frameon": False,
})

def style_ax(ax, xgrid=False):
    ax.grid(axis="y" if not xgrid else "x")
    if xgrid:
        ax.grid(axis="y", visible=False)

def save_fig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"{name}.{ext}")
    print(f"saved {name}.pdf/.png -> {FIG_DIR.relative_to(ROOT)}")

RNG_SEED = 0
def boot_mean_ci(vals, n_boot=4000, seed=RNG_SEED):
    a = np.asarray(vals, dtype=float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(a), size=(n_boot, len(a)))
    means = a[idx].mean(axis=1)
    return a.mean(), *np.percentile(means, [2.5, 97.5])

def load_stage(arm, stage):
    path = ROOT / f"outputs/placement_ablation_v2_{arm}" / f"{stage}.jsonl"
    return [json.loads(l) for l in open(path) if l.strip()]

FIN, RET, FAN, RUNS = {}, {}, {}, {}
for arm in MODELS:
    FIN[arm] = load_stage(arm, "final_response_scores")
    RET[arm] = load_stage(arm, "retrieval_scores")
    FAN[arm] = load_stage(arm, "fanout_scores")
    RUNS[arm] = load_stage(arm, "runs")

def score_map(rows, metric):
    """(query_id, Vx) -> score for one arm/stage/metric."""
    return {(r["query_id"], VKEY[r["variant"]]): r["scores"][metric]
            for r in rows if metric in r.get("scores", {})}

def paired_contrast(sc, a, b, n_boot=4000, seed=RNG_SEED):
    qids = sorted({q for (q, v) in sc if v == a} & {q for (q, v) in sc if v == b})
    d = np.array([sc[(q, a)] - sc[(q, b)] for q in qids])
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))
    means = d[idx].mean(axis=1)
    lo, hi = np.percentile(means, [2.5, 97.5])
    return dict(mean=d.mean(), lo=lo, hi=hi, n=len(d), sig=(lo > 0) or (hi < 0))

META = {r["query_id"]: dict(task_type=r["task_type"], task_category=r["task_category"])
        for r in FIN["gemini-3.6-flash"]}
print("loaded", {a: len(FIN[a]) for a in MODELS})


loaded {'gemma-4-31b-it': 432, 'gemini-3.6-flash': 432, 'gemini-3.1-pro-preview': 432}


## 0 · Integrity & provenance

Every arm must show 432 runs (72×6), **zero** empty-evidence runs (the Tavily-incident gate), and
zero judge errors before any number below is trusted.

In [2]:

rows = []
for arm in MODELS:
    man = json.load(open(ROOT / f"outputs/placement_ablation_v2_{arm}/manifest.json"))
    empty = sum(1 for r in RUNS[arm] if not r.get("raw_search_results"))
    rows.append(dict(
        arm=MODELS[arm]["label"], runs=len(RUNS[arm]),
        empty_evidence=empty,
        judge_errors=sum(1 for st in (FIN, RET, FAN) for r in st[arm] if r.get("error")),
        agent_model=man["agent_model"], judge_model=man["judge_model"],
        finished_utc=man["finished_at_utc"],
    ))
integrity = pd.DataFrame(rows)
assert (integrity.runs == 432).all() and (integrity.empty_evidence == 0).all() \
       and (integrity.judge_errors == 0).all()
integrity


,arm,runs,empty_evidence,judge_errors,agent_model,judge_model,finished_utc
0,Gemma 4 31B-IT,432,0,0,gemma-4-31b-it,gemini-flash-latest,2026-08-02T22:38:06Z
1,Gemini 3.6 Flash,432,0,0,gemini-3.6-flash,gemini-flash-latest,2026-08-02T20:08:21Z
2,Gemini 3.1 Pro (prev.),432,0,0,gemini-3.1-pro-preview,gemini-flash-latest,2026-08-02T23:41:26Z


## 1 · Headline: intent satisfaction by placement, across models

The paper's Figure 1 candidate. Ordering is identical on all three models
(V0 < V1 < V2/V3 < V4≈V5); what differs is *which* single lever is stronger — see §2.

In [3]:

fig, ax = plt.subplots(figsize=(8.2, 4.2))
x = np.arange(len(V))
off = {arm: (i - 1) * 0.22 for i, arm in enumerate(MODELS)}
for arm, m in MODELS.items():
    sc = score_map(FIN[arm], "intent_satisfaction")
    means, los, his = [], [], []
    for v in V:
        vals = [s for (q, vv), s in sc.items() if vv == v]
        mu, lo, hi = boot_mean_ci(vals)
        means.append(mu); los.append(mu - lo); his.append(hi - mu)
    ax.errorbar(x + off[arm], means, yerr=[los, his], fmt="o", ms=6,
                color=m["color"], ecolor=m["color"], elinewidth=1.4, capsize=0,
                label=m["label"], zorder=3)
    ax.plot(x + off[arm], means, color=m["color"], lw=1.0, alpha=0.45, zorder=2)
    for xi, mu in zip(x + off[arm], means):
        ax.annotate(f"{mu:.2f}", (xi, mu), textcoords="offset points", xytext=(0, 7),
                    ha="center", fontsize=7.5, color=INK2)
ax.set_xticks(x, [VLAB[v] for v in V])
ax.set_ylabel("intent satisfaction (1–5, judge mean, 95% bootstrap CI)")
ax.set_ylim(1.8, 4.6)
ax.set_title("Personalization placement: final-answer intent satisfaction (72 items/point)")
ax.legend(loc="upper left", fontsize=9)
style_ax(ax)
save_fig(fig, "c1_intent_by_variant")
plt.show()


saved c1_intent_by_variant.pdf/.png -> reports/paper/figures


/var/folders/f0/mj5kb1qx7_n8884hb9gr_8mw0000gn/T/ipykernel_88783/66014064.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2 · The two channels: evidence-carried vs synthesis-carried personalization

**The core theoretical figure.** The *evidence channel* (V3−V1: persona only in queries) is
model-invariant at ≈ +0.75 — selection is externalized to the search engine, so any model
benefits equally. The *synthesis channel* (V2−V1: persona only at answer-writing) scales with
model capability — a stronger model can do the persona-selection from generic evidence itself.

In [4]:

arms = list(MODELS)
ch = {"evidence channel  (V3−V1)": ("V3", "V1", CH_EVIDENCE),
      "synthesis channel  (V2−V1)": ("V2", "V1", CH_SYNTH)}
fig, ax = plt.subplots(figsize=(8.6, 4.2))
x = np.arange(len(arms))
for name, (a, b, color) in ch.items():
    res = [paired_contrast(score_map(FIN[arm], "intent_satisfaction"), a, b) for arm in arms]
    y = [r["mean"] for r in res]
    ax.errorbar(x, y, yerr=[[r["mean"] - r["lo"] for r in res], [r["hi"] - r["mean"] for r in res]],
                fmt="o-", ms=7, lw=1.8, color=color, ecolor=color, elinewidth=1.3,
                capsize=0, label=name)
    ax.annotate(name, (x[-1] + 0.06, y[-1]), fontsize=9, color=color, va="center")
    for xi, r in zip(x, res):
        ax.annotate(f"{r['mean']:+.2f}", (xi, r["mean"]),
                    textcoords="offset points", xytext=(-10, 8), ha="right",
                    fontsize=8, color=INK2)
ax.axhline(0, color=INK2, lw=0.8)
ax.set_xticks(x, [MODELS[a]["short"] for a in arms])
ax.set_xlim(-0.4, len(arms) - 0.1 + 1.3)
ax.set_ylabel("Δ intent vs V1 generic fanout (paired, 95% CI)")
ax.set_title("Evidence channel is model-invariant; synthesis channel scales with capability")
style_ax(ax)
save_fig(fig, "c1_two_channels")
plt.show()


saved c1_two_channels.pdf/.png -> reports/paper/figures


/var/folders/f0/mj5kb1qx7_n8884hb9gr_8mw0000gn/T/ipykernel_88783/789141349.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3 · Full contrast forest

All headline paired contrasts with 95% bootstrap CIs. Filled = CI excludes zero; open = n.s.
Note V3−V2 flips sign at the frontier while everything else keeps direction.

In [5]:

CONTRASTS = [("V2", "V1"), ("V3", "V1"), ("V3", "V2"), ("V4", "V3"), ("V4", "V2"), ("V5", "V4")]
fig, ax = plt.subplots(figsize=(7.6, 5.4))
yt, yl = [], []
row = 0
for a, b in CONTRASTS:
    for i, arm in enumerate(MODELS):
        r = paired_contrast(score_map(FIN[arm], "intent_satisfaction"), a, b)
        y = -(row * 4 + i)
        c = MODELS[arm]["color"]
        ax.errorbar([r["mean"]], [y], xerr=[[r["mean"] - r["lo"]], [r["hi"] - r["mean"]]],
                    fmt="o", ms=6.5, color=c, ecolor=c, elinewidth=1.5, capsize=0,
                    markerfacecolor=c if r["sig"] else "white", markeredgewidth=1.4)
    yt.append(-(row * 4 + 1)); yl.append(f"{a} − {b}")
    row += 1
ax.axvline(0, color=INK2, lw=0.9)
ax.set_yticks(yt, yl)
ax.set_xlabel("Δ intent satisfaction (paired over 72 items, 95% bootstrap CI)")
ax.set_title("Placement contrasts across models  (filled = significant)")
handles = [plt.Line2D([], [], marker="o", ls="", color=m["color"], label=m["label"])
           for m in MODELS.values()]
ax.legend(handles=handles, loc="lower right", fontsize=8.5)
style_ax(ax, xgrid=True)
save_fig(fig, "c1_contrast_forest")
plt.show()


saved c1_contrast_forest.pdf/.png -> reports/paper/figures


/var/folders/f0/mj5kb1qx7_n8884hb9gr_8mw0000gn/T/ipykernel_88783/4084859497.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4 · Mechanism: persona in the queries → persona-fit evidence

Retrieval-judge metrics step up exactly when the persona enters the fanout (V3/V4), on every
model; V5 sits between because only some of its branches are persona-conditioned. This is the
causal link behind the evidence channel.

In [6]:

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8), sharex=True)
for ax, metric, title in zip(axes, ("result_persona_fit", "constraint_coverage"),
                             ("evidence persona-fit", "evidence constraint coverage")):
    x = np.arange(len(V))
    for arm, m in MODELS.items():
        sc = score_map(RET[arm], metric)
        means, err = [], []
        for v in V:
            vals = [s for (q, vv), s in sc.items() if vv == v]
            mu, lo, hi = boot_mean_ci(vals)
            means.append(mu); err.append((mu - lo, hi - mu))
        ax.errorbar(x + (list(MODELS).index(arm) - 1) * 0.2, means,
                    yerr=np.array(err).T, fmt="o", ms=5.5, color=m["color"],
                    elinewidth=1.2, capsize=0, label=m["label"])
    ax.set_xticks(x, [VLAB[v] for v in V], fontsize=8)
    ax.set_title(title)
    ax.set_ylim(2.2, 4.6)
    style_ax(ax)
axes[0].set_ylabel("retrieval judge score (1–5, 95% CI)")
axes[0].legend(fontsize=8.5, loc="upper left")
fig.suptitle("Persona-conditioned queries retrieve persona-fit evidence (all models)", y=1.02)
save_fig(fig, "c1_retrieval_mechanism")
plt.show()


saved c1_retrieval_mechanism.pdf/.png -> reports/paper/figures


/var/folders/f0/mj5kb1qx7_n8884hb9gr_8mw0000gn/T/ipykernel_88783/4045629576.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5 · The task-type taxonomy fails in both directions

The benchmark's declared retrieval-/synthesis-sensitive labels predict a crossover
(V3 wins retrieval-sensitive, V2 wins synthesis-sensitive). It never appears: the flash arm's
V3 edge concentrates on declared-**synthesis** items; the pro arm's V2 edge is significant on
declared-**retrieval** items. Placement effects are a property of the *model*, not the item label.

In [7]:

fig, ax = plt.subplots(figsize=(7.4, 3.9))
groups = ["retrieval_sensitive", "synthesis_sensitive"]
x0 = {g: i for i, g in enumerate(groups)}
for i, arm in enumerate(MODELS):
    sc = score_map(FIN[arm], "intent_satisfaction")
    for g in groups:
        qids = {q for q in META if META[q]["task_type"] == g}
        sub = {k: v for k, v in sc.items() if k[0] in qids}
        r = paired_contrast(sub, "V3", "V2")
        x = x0[g] + (i - 1) * 0.22
        c = MODELS[arm]["color"]
        ax.errorbar([x], [r["mean"]], yerr=[[r["mean"] - r["lo"]], [r["hi"] - r["mean"]]],
                    fmt="o", ms=7, color=c, elinewidth=1.5, capsize=0,
                    markerfacecolor=c if r["sig"] else "white", markeredgewidth=1.5,
                    label=MODELS[arm]["label"] if g == groups[0] else None)
ax.axhline(0, color=INK2, lw=0.9)
ax.set_xticks(list(x0.values()), ["declared retrieval-sensitive\n(n=36)",
                                  "declared synthesis-sensitive\n(n=36)"])
ax.set_ylabel("V3 − V2  Δ intent (paired, 95% CI)")
ax.set_title("No predicted crossover: labels don't govern which placement wins\n(filled = significant; >0 favors persona-in-queries)")
ax.legend(fontsize=8.5, loc="lower left")
style_ax(ax)
save_fig(fig, "c1_taxonomy_test")
plt.show()


saved c1_taxonomy_test.pdf/.png -> reports/paper/figures


/var/folders/f0/mj5kb1qx7_n8884hb9gr_8mw0000gn/T/ipykernel_88783/3177962640.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6 · Metric profile heatmaps (per model)

Direction-harmonized final-answer metrics (`overpersonalization` shown flipped as
`no_overpersonalization` so higher = better everywhere). Judge-artifact metrics
(`disconfirming_coverage`, `unsafe_or_overpersonalized_retrieval_risk`) are excluded — their
anchors are inconsistent across variants (documented in the paper's limitations). The
groundedness block is uniformly low because every arm runs the base, citation-free synthesizer
(V0 reads higher only because a single-search answer strays less).

In [8]:

METRICS = ["intent_satisfaction", "personalization_target_use", "non_genericness",
           "specificity", "missing_constraint_awareness", "actionability_without_overclaiming",
           "safety", "uncertainty_calibration", "unsupported_claim_risk",
           "no_overpersonalization", "groundedness", "citation_support"]
fig, axes = plt.subplots(1, 3, figsize=(12.6, 4.6), sharey=True)
for ax, (arm, m) in zip(axes, MODELS.items()):
    grid = np.zeros((len(METRICS), len(V)))
    for j, v in enumerate(V):
        for i, metric in enumerate(METRICS):
            src = "overpersonalization" if metric == "no_overpersonalization" else metric
            sc = score_map(FIN[arm], src)
            vals = [s for (q, vv), s in sc.items() if vv == v]
            mu = float(np.mean(vals))
            grid[i, j] = 6 - mu if metric == "no_overpersonalization" else mu
    im = ax.imshow(grid, cmap=SEQ, vmin=1, vmax=5, aspect="auto")
    for i in range(len(METRICS)):
        for j in range(len(V)):
            ax.text(j, i, f"{grid[i,j]:.1f}", ha="center", va="center", fontsize=7,
                    color="white" if grid[i, j] > 3.4 else INK)
    ax.set_xticks(range(len(V)), V)
    ax.set_title(m["label"], fontsize=10)
    ax.grid(False)
axes[0].set_yticks(range(len(METRICS)), METRICS, fontsize=8)
cb = fig.colorbar(im, ax=axes, shrink=0.75, pad=0.015)
cb.set_label("judge mean (1–5, higher = better)", fontsize=8.5)
fig.suptitle("Final-answer metric profiles by placement (direction-harmonized)", y=1.0)
save_fig(fig, "c1_metric_heatmap")
plt.show()


saved c1_metric_heatmap.pdf/.png -> reports/paper/figures


/var/folders/f0/mj5kb1qx7_n8884hb9gr_8mw0000gn/T/ipykernel_88783/2636255786.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7 · V2 vs V3 at item level: mostly ties, strong effects lopsided

Per-item intent gap distribution. At single-seed resolution half the items cannot distinguish
the two placements; among deciders, large gaps (|Δ|≥2) are predominantly V3-favoring on the
flash/gemma arms and split on pro — the basis for reporting *revealed* sensitivity as a
continuous, model-dependent property rather than an item label.

In [9]:

fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.2), sharey=True)
bins = np.arange(-4.5, 5.0, 1.0)
for ax, (arm, m) in zip(axes, MODELS.items()):
    sc = score_map(FIN[arm], "intent_satisfaction")
    qids = sorted({q for (q, v) in sc})
    d = [sc[(q, "V3")] - sc[(q, "V2")] for q in qids]
    counts, edges = np.histogram(d, bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    colors = [MUTED if abs(c) < 0.5 else m["color"] for c in centers]
    ax.bar(centers, counts, width=0.86, color=colors)
    for c_, n_ in zip(centers, counts):
        if n_:
            ax.annotate(str(n_), (c_, n_), textcoords="offset points", xytext=(0, 3),
                        ha="center", fontsize=7.5, color=INK2)
    ax.set_title(m["label"], fontsize=10)
    ax.set_xlabel("V3 − V2 per item")
    style_ax(ax)
axes[0].set_ylabel("items (n=72)")
fig.suptitle("Gray = ties; color = deciding items  (>0 favors persona-in-queries)", y=1.04)
save_fig(fig, "c1_v2v3_gap_dist")
plt.show()


saved c1_v2v3_gap_dist.pdf/.png -> reports/paper/figures


/var/folders/f0/mj5kb1qx7_n8884hb9gr_8mw0000gn/T/ipykernel_88783/4173972352.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8 · Paper tables

Contrast table (all arms × all headline contrasts, paired bootstrap CIs) exported as CSV and
LaTeX, plus the variant×model intent summary.

In [10]:

rows = []
for arm in MODELS:
    sc = score_map(FIN[arm], "intent_satisfaction")
    for a, b in CONTRASTS:
        r = paired_contrast(sc, a, b)
        rows.append(dict(model=MODELS[arm]["label"], contrast=f"{a}-{b}",
                         delta=round(r["mean"], 2), ci_lo=round(r["lo"], 2),
                         ci_hi=round(r["hi"], 2), significant=r["sig"]))
contrasts_df = pd.DataFrame(rows)
contrasts_df.to_csv(TAB_DIR / "c1_contrasts.csv", index=False)
(TAB_DIR / "c1_contrasts.tex").write_text(
    contrasts_df.to_latex(index=False, escape=True,
                          caption="C1 placement contrasts (paired bootstrap, 95\\% CI).",
                          label="tab:c1-contrasts"))
summary = pd.DataFrame({MODELS[arm]["label"]:
    {v: round(float(np.mean([s for (q, vv), s in score_map(FIN[arm], "intent_satisfaction").items() if vv == v])), 2)
     for v in V} for arm in MODELS}).T
summary.to_csv(TAB_DIR / "c1_intent_summary.csv")
print("wrote c1_contrasts.csv/.tex and c1_intent_summary.csv ->", TAB_DIR.relative_to(ROOT))
display(summary)
contrasts_df


wrote c1_contrasts.csv/.tex and c1_intent_summary.csv -> reports/paper/tables


,V0,V1,V2,V3,V4,V5
Gemma 4 31B-IT,2.12,2.62,3.11,3.35,3.82,3.69
Gemini 3.6 Flash,2.35,2.79,3.36,3.58,3.97,3.90
Gemini 3.1 Pro (prev.),2.21,2.64,3.65,3.40,4.03,4.08


,model,contrast,delta,ci_lo,ci_hi,significant
0,Gemma 4 31B-IT,V2-V1,0.49,0.32,0.67,True
1,Gemma 4 31B-IT,V3-V1,0.73,0.49,0.98,True
2,Gemma 4 31B-IT,V3-V2,0.24,-0.01,0.50,False
3,Gemma 4 31B-IT,V4-V3,0.47,0.28,0.67,True
4,Gemma 4 31B-IT,V4-V2,0.71,0.44,0.99,True
5,Gemma 4 31B-IT,V5-V4,-0.12,-0.31,0.04,False
6,Gemini 3.6 Flash,V2-V1,0.57,0.38,0.76,True
7,Gemini 3.6 Flash,V3-V1,0.79,0.54,1.04,True
8,Gemini 3.6 Flash,V3-V2,0.22,-0.00,0.46,False
9,Gemini 3.6 Flash,V4-V3,0.38,0.19,0.59,True


## Figure inventory → paper mapping

| file | suggested use |
|---|---|
| `c1_intent_by_variant` | Fig 1 — headline placement result |
| `c1_two_channels` | Fig 2 — evidence channel constant, synthesis channel capability-scaled |
| `c1_contrast_forest` | Fig 3 or appendix — full stats backbone |
| `c1_retrieval_mechanism` | Fig 4 — mechanism (persona-fit evidence) |
| `c1_taxonomy_test` | benchmark-validity section — two-sided label falsification |
| `c1_metric_heatmap` | appendix — full metric profiles |
| `c1_v2v3_gap_dist` | benchmark-validity section — tie mass / revealed sensitivity |

**Caveats to carry into the paper:** single seed per arm; `gemini-3.1-pro-preview` is a preview
checkpoint; judge fixed at `gemini-flash-latest` (floating alias — all arms scored in the same
window); groundedness family reflects the base citation-free synthesizer; two retrieval-judge
metrics carry known anchor artifacts and are excluded above; the `gemini-3.5-flash` baseline arm
(bridge to C2/C3) is still pending and the two-channel framing predicts its V2−V1 lands between
Gemma's and 3.6-Flash's.